# Getting started


```{note}

Feel free to click through the individual tutorials, some of them are already finished!
```

```{warning}

This **"Getting started"** tutorial will give a brief general overview and then link to the specific tutorials on each major concept. It is still work-in-progress and will be finalized as the other more specific tutorials mature.
```

## 1. The `xsnowDataset` data model
Unlike data models that treat each *single snow profile* as a standalone object, `xsnow` organizes data on a labeled grid. That means one dataset can hold:

- multiple locations,
- multiple times,
- multiple slope/aspect configurations per location and time,
- multiple realizations (e.g. different weather forcings, ensemble members), and
- each full stratigraphy (layered snow profile) underneath those.

Formally, the core dimensions are:

1. **`location`**
   Where is this profile? We typically attach geographic metadata as *coordinates* on this dimension (e.g. `latitude`, `longitude`, `easting`, `northing`, `altitude`).

2. **`time`**
   When is this profile valid?
   Note: `time` is *timezone-naive* and reflects whatever the data source used. How to handle time zones correctly is covered in [Handling timezones](./timezones).

3. **`slope`**
   Same physical location and time, but different terrain orientation. xsnow attaches the coordinates
     - `inclination` (degrees from horizontal)
     - `azimuth` (degrees from north; e.g., 180° = south)
     
     If you only have one slope/aspect, the dimension length is 1.
   

4. **`realization`**
   Parallel "versions" of the snowpack for the same `(location, time, slope)` — for example:
   - different weather inputs (multi model, or ensemble model members),
   - different physical parametrizations,
   - observed vs. modeled,
   - different land cover classes, etc.  
   If you don't use this dimension, it's still present but length 1.
   
5. **`layer`**
   The vertical snow stratigraphy. Variables defined along `layer` describe per-layer properties: density, grain size, temperature, etc. There is also a vertical coordinate variable like `z` (zero at the snow surface, positive upward, negative downward).


Those five dimensions give you a consistent, rectangular grid. On top of them, we store two broad kinds of variables:

| Kind of variable                          | Typical dims                                             |
|------------------------------------------|----------------------------------------------------------|
| **Profile-level / scalar per profile**     | `(location, time, slope, realization)`                   |
| **Layer-level / stratigraphy variable** | `(location, time, slope, realization, layer)`            |

Example:
- `HS` (total snow height) is profile-level and therefore does not have a `layer` dimension.
- `density` is a layer property and therefore it *does* have the `layer` dimension.


**Why this structure?**

- You can run fully vectorized analysis and statistics across large subsets of the data.  
  For example: *“Give me all weak layers with poor stability on north-facing slopes after a certain date”* becomes a straightforward selection and filter on the dataset---no manual loops, no filename parsing, no ad hoc per-station scripts.

- The entire dataset can be serialized to (and restored from) NetCDF efficiently, lazily, and it can be subset on parsing.

- By adopting this data model, `xsnow` can directly leverage the high-performance [`xarray`](https://docs.xarray.dev/en/stable/) ecosystem.  
  This not only gives us convenient labeled indexing, resampling, and computation, but also sets up `xsnow` for future-friendly storage formats like *zarr* / *geozarr*, which are increasingly used for scalable, cloud-native geoscience data.


```{admonition} An xsnowDataset feels like an xarray.Dataset
:class: note

While `xsnow` introduces its own class, `xsnowDataset`, we’ve taken meticulous care to make an `xsnowDataset` *behave and feel* just like an `xarray.Dataset`.

- **Full compatibility:**  
  Any method you can call on an `xarray.Dataset` can also be called on an `xsnowDataset`.

- **Underlying xarray object:**  
  Every `xsnowDataset` contains a regular `xarray.Dataset` holding the actual data, accessible via the attribute `ds.data`.  
  If you want to explicitly export to a pure `xarray.Dataset`, simply call:

      ds.to_xarray()

- **Automatic method forwarding:**  
  Any attribute or method that’s not part of `xsnowDataset` is automatically forwarded to the underlying `xarray.Dataset`.  
  For example:

      ds.info()      # under the hood calls ds.data.info()
      ds['density']  # under the hood calls ds.data['density']

- **Helpful auto-complete organization:**  
  Python’s tab completion works seamlessly and is structured to help you explore:

      ds.       # + TAB: shows xsnow-native methods and coordinates
      ds.data.  # + TAB: shows all xarray.Dataset methods available on ds as well

This design keeps the workflow intuitive: you get all the power of `xarray`, plus the specialized snow science functionality of `xsnow`.
```

```{warning}

More coming soon!
```